# GDP Nowcasting — Exploration Notebook

Use this notebook to explore the data, diagnose the models, and develop intuitions before committing logic to `.py` files.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

from config import INDICATORS, TARGET_SERIES_ID, INDICATOR_IDS

## 1. Load data

In [ ]:
# First run — fetches from FRED (requires FRED_API_KEY)
# import os; os.environ['FRED_API_KEY'] = 'your_key_here'

from data.ingest import ingest
panel = ingest(save=True)
print(f'Panel shape: {panel.shape}')
print(f'Date range: {panel.index[0].date()} to {panel.index[-1].date()}')
panel.tail()

## 2. Inspect the ragged edge

This is the core challenge of nowcasting — not all series are available at the same time.

In [ ]:
SERIES_NAMES = {s.fred_id: s.name for s in INDICATORS}

last_obs = panel.apply(lambda col: col.last_valid_index())
last_obs_df = pd.DataFrame({'series': last_obs.index, 'last_obs': last_obs.values})
last_obs_df['name'] = last_obs_df['series'].map(SERIES_NAMES)
last_obs_df = last_obs_df.sort_values('last_obs', ascending=False)

print('Last available observation by series:')
print(last_obs_df[['name', 'last_obs']].to_string(index=False))

## 3. Correlations with GDP

In [ ]:
from models.bridge import to_quarterly

quarterly = to_quarterly(panel)
target = quarterly[TARGET_SERIES_ID]

corrs = {}
for ind in INDICATOR_IDS:
    if ind in quarterly.columns:
        corr = quarterly[ind].corr(target)
        corrs[SERIES_NAMES.get(ind, ind)] = round(corr, 3)

corr_df = pd.Series(corrs).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_df.values]
corr_df.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with quarterly GDP growth')
ax.set_title('Indicator correlations with GDP (quarterly)')
plt.tight_layout()
plt.savefig('../output/charts/indicator_correlations.png', dpi=150)
plt.show()

## 4. Fit bridge model and inspect equations

In [ ]:
from models.bridge import BridgeModel, get_quarter_to_date_means

bridge = BridgeModel()
bridge.fit(quarterly)

print('Bridge equation summary (sorted by R²):')
summary = bridge.summary()
summary['name'] = summary['indicator'].map(SERIES_NAMES)
print(summary[['name', 'r2', 'coef', 'intercept']].to_string(index=False))

In [ ]:
# Generate current nowcast
qtd_means = get_quarter_to_date_means(panel, panel.index[-1])
result = bridge.predict(qtd_means)

print(f"\nCurrent nowcast: {result['nowcast']:+.2f}% QoQ annualized")
print(f"Based on {result['n_equations']} bridge equations")
print("\nContributions:")
for ind, val in sorted(result['components'].items(), key=lambda x: -abs(x[1])):
    print(f"  {SERIES_NAMES.get(ind, ind):<40} {val:+.3f}pp")

## 5. DFM (optional — takes a few minutes)

In [ ]:
# Uncomment when ready to run the DFM
# from models.dfm import DFMNowcaster
#
# dfm = DFMNowcaster()
# dfm.fit(panel)
# dfm_result = dfm.nowcast(panel)
# print(f'DFM nowcast: {dfm_result["nowcast"]:+.2f}%')
#
# # Plot factor dynamics
# dfm_result['factors'].plot(figsize=(10,4), title='Estimated common factors')
# plt.tight_layout()
# plt.show()

## 6. Nowcast history (in-sample fitted values)

In [ ]:
gdp_actual = quarterly[TARGET_SERIES_ID].dropna()

fitted_vals = {}
for ind, model in bridge.models.items():
    x = quarterly[ind].dropna()
    x_scaled = bridge.scalers[ind].transform(x.values.reshape(-1,1))
    fitted_vals[ind] = pd.Series(model.predict(x_scaled), index=x.index)

ensemble_fitted = pd.DataFrame(fitted_vals).mean(axis=1)
ensemble_fitted = ensemble_fitted[ensemble_fitted.index.isin(gdp_actual.index)]

fig, ax = plt.subplots(figsize=(12, 5))
gdp_actual.plot(ax=ax, color='#1D9E75', linewidth=2, label='Actual GDP growth')
ensemble_fitted.plot(ax=ax, color='#534AB7', linewidth=1.5, linestyle='--', label='Bridge model (in-sample)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('% QoQ annualized')
ax.set_title('Bridge model: fitted vs. actual GDP growth')
ax.legend()
plt.tight_layout()
plt.savefig('../output/charts/bridge_fitted_vs_actual.png', dpi=150)
plt.show()

errors = (gdp_actual - ensemble_fitted).dropna()
rmse = np.sqrt((errors**2).mean())
print(f'In-sample RMSE: {rmse:.3f}pp')